### **DISCLAIMER: Due to the topic of bias and fairness, some users may be offended by the content contained herein, including prompts and output generated from use of the prompts.**

### Import packages, define constants, set up credentials

In [2]:
import openai
import torch
from dotenv import find_dotenv, load_dotenv
from langchain_core.rate_limiters import InMemoryRateLimiter

from langfair.generator import AdversarialGenerator
from langfair.metrics.toxicity import ToxicityMetrics

### Toxicity Redteaming Evaluation

#### Adversarial Toxicity Generation
***
##### `AdversarialGenerator()` - Class for generating evaluation datasets for adversarial model-level assessments. This class offers two methods: `counterfactual()` and `toxicity()`, which generate evaluation datasets required for respective assessments. For more on these approaches, refer to https://arxiv.org/pdf/2306.11698 (class)

##### Class parameters:

- `langchain_llm` (**langchain llm (Runnable), default=None**) A langchain llm object to get passed to LLMChain `llm` argument. Must be provided if `custom_llm_object` is `None`.
- `suppressed_exceptions` : (**tuple, default=None**) Specifies which exceptions to handle as 'Unable to get response' rather than raising the exception

##### Methods:
***
##### `toxicity()` -  Generates toxicity dataset using either benign or adversarial system prompt. If using RealToxicityPrompts, user can also specify whether to use toxic or nontoxic prompts as well as sample size. Instead of RealToxicityPrompts, the user may instead choose to supply their own custom list of prompts to create generations for toxicity evaluations. For each prompt, `count` responses are generated. 
###### Method Parameters:

- `system_style` - (**{'benign', 'adversarial', 'custom'}, default='benign'**) Specifies whether to use benign, adversarial, or custom system prompt
- `prompt_style` - (**{'toxic', 'nontoxic'}, default='toxic'**) Specifies whether to use toxic or nontoxic task prompts (if using RealToxicityPrompts) or custom user prompts (if not using RealToxicityPrompts)  
- `sample_size` - (**int, default=100**) Specifies how many rows to sample from RealToxicityPrompts
- `custom_system_prompt` - (**str or None, default=None**) Optional argument for user to provide custom system prompt for toxicity generation if not using RealToxicityPrompts
- `sampling_seed` - (**int, default=123**) For setting the random state when sampling from RealToxicityPrompts
- `count` - (**int, default=25**) Specifies number of responses to generate for each prompt. The convention is to use 25 generations per prompt in evaluating toxicity. See, for example DecodingTrust (https://arxiv.org/abs//2306.11698) or HELM (https://arxiv.org/abs/2211.09110)

###### Returns:
A dictionary with two keys: `data` and `metadata`.
- `data` (**dict**) A dictionary containing the prompts and responses.
- `metadata` (**dict**) A dictionary containing metadata about the generation process, including non-completion rate, temperature, and count.

###### Instantiate adversarial generator class

In [3]:
# Use LangChain's InMemoryRateLimiter to avoid rate limit errors. Adjust parameters as necessary.
rate_limiter = InMemoryRateLimiter(
    requests_per_second=10,
    check_every_n_seconds=10,
    max_bucket_size=1000,
)

In [4]:
# Run if langchain-openai not installed
# import sys
# !{sys.executable} -m pip install langchain-openai
from langchain_openai import AzureChatOpenAI

# User to populate .env file with API credentials
load_dotenv(find_dotenv())

# Setup langchain llm object
llm = AzureChatOpenAI(
    deployment_name="gpt-4o",
    openai_api_type="azure",
    openai_api_version="2024-02-15-preview",
    temperature=1,  # User to set temperature
    rate_limiter=rate_limiter,
)

In [5]:
ag = AdversarialGenerator(
    langchain_llm=llm,
    suppressed_exceptions=(
        openai.BadRequestError,
        ValueError,
    ),  # suppresses content filtering errors
)

###### Generate toxicity data using benign system prompt

In [6]:
# With toxic task prompts
benign_toxic = await ag.toxicity(
    system_style="benign",
    prompt_style="toxic",
    sample_size=1000,  # 1000 is the recommended sample_size
)

Generating 25 responses per prompt...
Responses successfully generated!


In [14]:
# With nontoxic task prompts
benign_nontoxic = await ag.toxicity(
    system_style="benign",
    prompt_style="nontoxic",
    sample_size=1000,  # 1000 is the recommended sample_size
)

Generating 25 responses per prompt...
Responses successfully generated!


###### Generate toxicity data using adversarial system prompt

In [15]:
# With toxic task prompts
adversarial_toxic = await ag.toxicity(
    system_style="adversarial",
    prompt_style="toxic",
    sample_size=1000,  # 1000 is the recommended sample_size
)

Generating 25 responses per prompt...


In [9]:
# With nontoxic task prompts
adversarial_nontoxic = await ag.toxicity(
    system_style="adversarial",
    prompt_style="nontoxic",
    sample_size=1000,  # 1000 is the recommended sample_size
)

Generating 25 responses per prompt...
Responses successfully generated!


#### Calculate Toxicity Metrics
***


##### `ToxicityMetrics()` - For calculating the toxicity bias metrics (class)
**Class Attributes:**

- `classifiers` - (*list containing subset of {'detoxify_unbiased', detoxify_original, 
        'roberta-hate-speech-dynabench-r4-target','toxigen'}, 
            default = ['detoxify_unbiased'])
            Specifies which toxicity classifier to use.
- `toxic_threshold` - (**float, default=0.5**) Specifies which threshold to use when binarizing toxicity probabilities.
- `batch_size` - (**int, default=250**) Specifies the batch size for scoring toxicity of texts. Avoid setting too large to prevent the kernel from dying.
- `device` - (str or torch.device input or torch.device object, default="cpu")
            Specifies the device that classifiers use for prediction. Set to "cuda" for classifiers to be able to leverage the GPU. 
            Currently, 'detoxify_unbiased' and 'detoxify_original' will use this parameter. 

**Methods:**
1. `get_toxicity_scores()` - Calculates expected maximum toxicity metric.
    Method Parameters:
    - `texts` - (**List of strings**) A list of texts to be scored with a toxicity classifierbenign_toxic

    Returns:
    - vector of toxicity probabilities (**List of floats**)

2. `evaluate_metrics()` - Calculate toxicity scores and calculate toxic fraction, expected maximum toxicity, and toxicity probability metrics.
    Method Parameters:

    - `responses` - (**List of strings**) A list of generated output from an LLM
    - `prompts` - (**List of strings, default=None**) A list of prompts from which `responses` were generated. Only required if `by_prompt=True`.
    - `return_df` - (**Bool, default=True**) Specifies whether to return a dataframe containing response-level toxicity scores
    - `by_prompt` - (**Bool, default=False**) Specifies whether metrics should be calculated by prompt and averaged across prompts or applied as a single calculation over all responses. If `False`, only toxic fraction is calculated. 

In [7]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)  # Use GPU if available
tm = ToxicityMetrics(classifiers=["detoxify_unbiased"], batch_size=100, device=device)

/Users/c594266/langfair-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Evaluate toxicity metrics for responses generated with benign system prompts

In [8]:
# Benign system prompts with toxic user prompts
benign_toxic_results = tm.evaluate(
    responses=list([str(x) for x in benign_toxic["data"]["response"]]),
    prompts=benign_toxic["data"]["prompt"],
)
benign_toxic_results

Computing toxicity scores...
Evaluating metrics...


{'metrics': {'Toxic Fraction': 0.0,
  'Expected Maximum Toxicity': 0.11267144232988358,
  'Toxicity Probability': 0}}

In [16]:
# Benign system prompts with nontoxic user prompts
benign_nontoxic_results = tm.evaluate(
    responses=list([str(x) for x in benign_nontoxic["data"]["response"]]),
    prompts=benign_nontoxic["data"]["prompt"],
)
benign_nontoxic_results

Computing toxicity scores...
Evaluating metrics...


{'metrics': {'Toxic Fraction': 0.0,
  'Expected Maximum Toxicity': 0.00251021608710289,
  'Toxicity Probability': 0}}

Evaluate toxicity metrics for responses generated with adversarial system prompts (only needed if system prompt can be modified in use case)

In [17]:
# Adversarial system prompts with toxic user prompts
adversarial_toxic_results = tm.evaluate(
    responses=list([str(x) for x in adversarial_toxic["data"]["response"]]),
    prompts=adversarial_toxic["data"]["prompt"],
)
adversarial_toxic_results

Computing toxicity scores...
Evaluating metrics...


{'metrics': {'Toxic Fraction': 0.42000000000000004,
  'Expected Maximum Toxicity': 0.9891928136348724,
  'Toxicity Probability': 1}}

In [18]:
# Adversarial system prompts with nontoxic user prompts
adversarial_nontoxic_results = tm.evaluate(
    responses=list([str(x) for x in adversarial_nontoxic["data"]["response"]]),
    prompts=adversarial_nontoxic["data"]["prompt"],
)
adversarial_nontoxic_results

Computing toxicity scores...
Evaluating metrics...


{'metrics': {'Toxic Fraction': 0.8,
  'Expected Maximum Toxicity': 0.9968747496604919,
  'Toxicity Probability': 1}}